### ***Shahid Patel | 31 | 232A004***

***Experiment No.5: Implementing Monte Carlo control and Temporal Difference (TD) learning algorithms to train an agent in a grid-world environment.***

***1.Import Librabries***

In [64]:
print('Shahid Patel | 31 | 232A004')
import numpy as np
import random

#GridWorld Environment
class GridWorld:
    def __init__(self, size=4, start=(0,0), goal=(3,3)):
        self.size = size
        self.start = start
        self.goal = goal
        self.state = start
        self.actions = [(0,1), (1,0), (0,-1), (-1,0)]

    def reset(self):
        self.state = self.start
        return self.state

    def step(self,action):
      next_state = (self.state[0] + self.actions[action][0],
                    self.state[1] + self.actions[action][1])

      if 0 <= next_state[0] < self.size and 0 <= next_state[1] < self.size:
        self.state = next_state
        reward = 1 if self.state == self.goal else -0.1
        done = self.state == self.goal
      else:
        # If action leads out of bounds, stay in current state and penalize
        reward = -1 # Penalty for hitting a wall
        done = False # Episode is not over

      return self.state, reward, done

Shahid Patel | 31 | 232A004


***2.Monte Carlo Approach***

In [65]:
print('Shahid Patel | 31 | 232A004')
def monte_carlo_control(env, episodes=5000, gamma=0.9, epsilon=0.1):
    # Initialize Q-table and returns dictionary
    Q = {((i, j), a): 0 for i in range(env.size) for j in range(env.size) for a in range(4)}
    returns = {((i, j), a): [] for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        episode = []

        while True:
            action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[state, a])
            next_state, reward, done = env.step(action)
            episode.append((state, action, reward))
            state = next_state
            if done:
                break

    G = 0
    for state, action, reward in reversed(episode):
        G = gamma * G + reward
        if (state, action) not in [(s, a) for s, a, _ in episode[:-1]]:
            returns[(state, action)].append(G)
            Q[(state, action)] = np.mean(returns[(state, action)])

    return Q

Shahid Patel | 31 | 232A004


***3.SARSA (State-Action-Reward-State-Action)***

In [66]:
print('Shahid Patel | 31 | 232A004')
# Temporal Difference Learning (SARSA)
def sarsa(env, episodes=5000, alpha=0.1, gamma=0.99, epsilon=0.1):
    Q = {((i, j), a): 0.0 for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()
        action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[state, a])

        while True:
            next_state, reward, done = env.step(action)
            next_action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[next_state, a])
            # Update Rule
            Q[state, action] += alpha * (reward + gamma * Q[next_state, next_action] - Q[state, action])
            state, action = next_state, next_action
            if done:
                break
        return Q

Shahid Patel | 31 | 232A004


***4. Q-Learning (Off-Policy TD Control)***
*Q-Learning is very similar to SARSA but uses max future Q-value instead of following the policy.*

In [67]:
print('Shahid Patel | 31 | 232A004')
# Q-Learning Algorithm
def q_learning(env, episodes=5000, alpha=0.1, gamma=0.99, epsilon=0.1):
    Q = {((i, j), a): 0.0 for i in range(env.size) for j in range(env.size) for a in range(4)}

    for _ in range(episodes):
        state = env.reset()

        while True:
            # Choose action using epsilon-greedy policy
            action = random.choice(range(4)) if random.random() < epsilon else max(range(4), key=lambda a: Q[state, a])
            next_state, reward, done = env.step(action)
            # Q-Learning Update Rule (using max over future actions)
            Q[state, action] += alpha * (reward + gamma * max(Q[next_state, a] for a in range(4)) - Q[state, action])
            state = next_state
            if done:
                break
        return Q

Shahid Patel | 31 | 232A004


5.

In [68]:
print('Shahid Patel | 31 | 232A004')
# Initialize environment
env = GridWorld()
# Train with Monte Carlo
Q_mc = monte_carlo_control(env)
# Train with SARSA
Q_sarsa = sarsa(env)
# Train with Q-Learning
Q_qlearning = q_learning(env)

Shahid Patel | 31 | 232A004


***6.Sample Q-Values***

*At the end,the script prints sample Q-values to compare the difference learning methods:*

In [69]:
print('Shahid Patel | 31 | 232A004')
print("Sample Q-values from Monte Carlo:", {k: Q_mc[k] for k in list(Q_mc.keys())[:20]})
print("Sample Q-values from SARSA:", {k: Q_sarsa[k] for k in list(Q_sarsa.keys())[:20]})
print("Sample Q-values from Q-Learning:", {k: Q_qlearning[k] for k in list(Q_qlearning.keys())[:20]})

Shahid Patel | 31 | 232A004
Sample Q-values from Monte Carlo: {((0, 0), 0): 0, ((0, 0), 1): 0, ((0, 0), 2): 0, ((0, 0), 3): 0, ((0, 1), 0): 0, ((0, 1), 1): 0, ((0, 1), 2): 0, ((0, 1), 3): 0, ((0, 2), 0): 0, ((0, 2), 1): 0, ((0, 2), 2): 0, ((0, 2), 3): 0, ((0, 3), 0): 0, ((0, 3), 1): 0, ((0, 3), 2): 0, ((0, 3), 3): 0, ((1, 0), 0): 0, ((1, 0), 1): 0, ((1, 0), 2): 0, ((1, 0), 3): 0}
Sample Q-values from SARSA: {((0, 0), 0): -0.010000000000000002, ((0, 0), 1): 0.0, ((0, 0), 2): 0.0, ((0, 0), 3): 0.0, ((0, 1), 0): -0.010000000000000002, ((0, 1), 1): 0.0, ((0, 1), 2): 0.0, ((0, 1), 3): 0.0, ((0, 2), 0): -0.010000000000000002, ((0, 2), 1): 0.0, ((0, 2), 2): 0.0, ((0, 2), 3): 0.0, ((0, 3), 0): -0.19, ((0, 3), 1): -0.010000000000000002, ((0, 3), 2): 0.0, ((0, 3), 3): 0.0, ((1, 0), 0): 0.0, ((1, 0), 1): 0.0, ((1, 0), 2): 0.0, ((1, 0), 3): 0.0}
Sample Q-values from Q-Learning: {((0, 0), 0): -0.010000000000000002, ((0, 0), 1): 0.0, ((0, 0), 2): 0.0, ((0, 0), 3): 0.0, ((0, 1), 0): -0.01000000000000